# Train the PCB defect detector, version 2 (with tiling)

**Project:** Bharat AI Hackathon, on-device PCB inspection on Raspberry Pi 5.

**Why a version 2:** in version 1 every board photo had been shrunk to 640 x 640 pixels, which left a
typical defect only about 14 pixels wide. The model could not learn from something that small (mAP50 was 0.04).

**What is different here:** we use the original full-size photos (about 3000 x 2000 pixels) and cut each one
into overlapping 640 x 640 pieces called **tiles**. Nothing is shrunk, so a defect stays about 60 pixels wide
and is easy to see. The model is trained on tiles, and later the Raspberry Pi will cut each photo into tiles
in the same way before inspecting it.

**Before you start:** menu **Runtime → Change runtime type → T4 GPU → Save**.
Run the cells from top to bottom with **Shift + Enter** and keep this tab open until the last cell finishes.

## Step 1: Check that the GPU is on
This should print a table that mentions **Tesla T4** (or another GPU).

In [ ]:
!nvidia-smi

## Step 2: Install the libraries
- `ultralytics` contains YOLO.
- `kagglehub` downloads datasets from Kaggle, a public data-science website.

In [ ]:
!pip install -q ultralytics kagglehub

## Step 3: Download the original full-size dataset
This is the same Peking University PCB dataset as before, but the untouched original:
693 photos with one label file each. It is about 2 GB, so the download takes a minute or two.
No account or key should be needed.

In [ ]:
import glob, os
import kagglehub

dataset_dir = kagglehub.dataset_download("akhatova/pcb-defects")
print("Dataset saved in:", dataset_dir)

# One label file (.xml) exists for every labelled photo.
xml_files = sorted(glob.glob(os.path.join(dataset_dir, "**", "*.xml"), recursive=True))

# Build a lookup from photo name to photo path. The dataset also has a "rotation" folder with
# rotated copies that have no label files, so we skip that folder.
image_paths = {}
for p in sorted(glob.glob(os.path.join(dataset_dir, "**", "*.jpg"), recursive=True)):
    if "rotation" in p.lower():
        continue
    image_paths.setdefault(os.path.splitext(os.path.basename(p))[0], p)

print("Label files found:", len(xml_files))
print("Photos found     :", len(image_paths))

## Step 4: Cut the photos into tiles
What this cell does, in order:

1. **Reads each label file.** It lists the defects in one photo as boxes (left, top, right, bottom in pixels).
2. **Splits the photos into three groups:** 80% train (to learn from), 10% valid (to check progress during
   training), 10% test (for the final score). We split whole photos, so tiles from one photo never end up in
   two groups. Otherwise the model would be tested on things it had already seen.
3. **Cuts every photo into 640 x 640 tiles** that overlap by 128 pixels. The overlap means a defect sitting
   on the edge of one tile is completely inside the neighbouring tile.
4. **Writes the label for each tile** in YOLO's format: class number, then the box centre, width and height
   as fractions of the tile size.
5. **Keeps only some of the empty tiles for training.** Most tiles contain no defect. We keep 15% of them so
   the model also learns what a clean area looks like, without drowning in empty examples.

In [ ]:
import random, shutil
import xml.etree.ElementTree as ET
from PIL import Image

CLASSES = ["missing_hole", "mouse_bite", "open_circuit", "short", "spur", "spurious_copper"]
TILE = 640            # tile width and height in pixels
STRIDE = 512          # how far we move for the next tile (640 - 512 = 128 pixels of overlap)
KEEP_EMPTY_TRAIN = 0.15
TILES_DIR = "/content/pcb_tiles"


def read_boxes(xml_path):
    """Returns the defects in one label file as a list of (class_number, left, top, right, bottom)."""
    boxes = []
    for obj in ET.parse(xml_path).getroot().iter("object"):
        name = obj.findtext("name").strip().lower().replace(" ", "_")
        if name not in CLASSES:
            continue
        b = obj.find("bndbox")
        left, top, right, bottom = [float(b.findtext(k)) for k in ("xmin", "ymin", "xmax", "ymax")]
        boxes.append((CLASSES.index(name), left, top, right, bottom))
    return boxes


def tile_starts(length):
    """Where each tile begins along one side of the photo, so the tiles cover it completely."""
    if length <= TILE:
        return [0]
    starts = list(range(0, length - TILE + 1, STRIDE))
    if starts[-1] != length - TILE:
        starts.append(length - TILE)      # one last tile flush with the far edge
    return starts


def boxes_in_tile(boxes, x0, y0, x1, y1):
    """Keeps the boxes that are at least half inside the tile and converts them to YOLO format."""
    lines = []
    for cls, left, top, right, bottom in boxes:
        il, it, ir, ib = max(left, x0), max(top, y0), min(right, x1), min(bottom, y1)
        if ir - il < 3 or ib - it < 3:
            continue                      # not inside this tile
        if (ir - il) * (ib - it) < 0.5 * (right - left) * (bottom - top):
            continue                      # less than half visible, a neighbouring tile has it whole
        w, h = x1 - x0, y1 - y0
        lines.append(f"{cls} {((il + ir) / 2 - x0) / w:.6f} {((it + ib) / 2 - y0) / h:.6f} "
                     f"{(ir - il) / w:.6f} {(ib - it) / h:.6f}")
    return lines


# 1. Read every label file and group the photos by defect class.
photos_by_class = {c: [] for c in range(len(CLASSES))}
labels = {}
for xml_path in xml_files:
    stem = os.path.splitext(os.path.basename(xml_path))[0]
    boxes = read_boxes(xml_path)
    if stem in image_paths and boxes:
        labels[stem] = boxes
        photos_by_class[boxes[0][0]].append(stem)

# 2. Split whole photos 80 / 10 / 10, class by class so every group has all six classes.
rng = random.Random(0)                    # fixed seed: the split is the same every time
split_of = {}
for stems in photos_by_class.values():
    stems = sorted(stems)
    rng.shuffle(stems)
    n_test = max(1, round(0.1 * len(stems)))
    for i, stem in enumerate(stems):
        split_of[stem] = "test" if i < n_test else "valid" if i < 2 * n_test else "train"

# 3-5. Cut the tiles and write their labels.
shutil.rmtree(TILES_DIR, ignore_errors=True)
counts = {s: {"photos": 0, "tiles": 0, "empty_tiles": 0, "boxes": 0} for s in ("train", "valid", "test")}
for split in counts:
    os.makedirs(os.path.join(TILES_DIR, split, "images"))
    os.makedirs(os.path.join(TILES_DIR, split, "labels"))

for stem, split in sorted(split_of.items()):
    photo = Image.open(image_paths[stem]).convert("RGB")
    W, H = photo.size
    counts[split]["photos"] += 1
    for y0 in tile_starts(H):
        for x0 in tile_starts(W):
            x1, y1 = min(x0 + TILE, W), min(y0 + TILE, H)
            lines = boxes_in_tile(labels[stem], x0, y0, x1, y1)
            if not lines and split == "train" and rng.random() > KEEP_EMPTY_TRAIN:
                continue
            name = f"{stem}_x{x0}_y{y0}"
            photo.crop((x0, y0, x1, y1)).save(os.path.join(TILES_DIR, split, "images", name + ".jpg"), quality=92)
            with open(os.path.join(TILES_DIR, split, "labels", name + ".txt"), "w") as f:
                f.write("\n".join(lines))
            counts[split]["tiles"] += 1
            counts[split]["boxes"] += len(lines)
            counts[split]["empty_tiles"] += 0 if lines else 1

yaml_path = os.path.join(TILES_DIR, "data.yaml")
with open(yaml_path, "w") as f:
    f.write(f"path: {TILES_DIR}\ntrain: train/images\nval: valid/images\ntest: test/images\nnames:\n")
    for i, c in enumerate(CLASSES):
        f.write(f"  {i}: {c}\n")

for split, c in counts.items():
    print(f"{split:5s}: {c['photos']:3d} photos -> {c['tiles']:5d} tiles "
          f"({c['empty_tiles']} without defects), {c['boxes']} defect boxes")

## Step 5: Look at a few tiles to check the labels
Before spending an hour on training, we draw the labels on four random training tiles.
Each red box should sit on a visible defect. If the boxes are clearly in the wrong places, stop and tell Claude.

In [ ]:
from PIL import ImageDraw
from IPython.display import display

label_files = [p for p in sorted(glob.glob(os.path.join(TILES_DIR, "train", "labels", "*.txt"))) if os.path.getsize(p) > 0]
sheet = Image.new("RGB", (TILE * 4, TILE), "white")
for i, label_path in enumerate(random.Random(1).sample(label_files, 4)):
    tile = Image.open(label_path.replace("labels", "images").replace(".txt", ".jpg")).convert("RGB")
    draw = ImageDraw.Draw(tile)
    for line in open(label_path):
        cls, cx, cy, w, h = line.split()
        cx, cy, w, h = float(cx) * tile.width, float(cy) * tile.height, float(w) * tile.width, float(h) * tile.height
        draw.rectangle([cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2], outline="red", width=3)
        draw.text((cx - w / 2, cy - h / 2 - 14), CLASSES[int(cls)], fill="red")
    sheet.paste(tile, (TILE * i, 0))
display(sheet.resize((TILE * 2, TILE // 2)))

## Step 6: Train
Same idea as before: start from the small pretrained `yolov8n.pt` and fine-tune it on our tiles.

- `epochs=60`: up to 60 passes over the training tiles.
- `patience=15`: stop early if the score has not improved for 15 epochs, to save time.
- `flipud=0.5`: half the time a tile is shown upside down, because a board can face any way.

This is the slow cell, roughly 1 to 1.5 hours. This time `mAP50` should climb quickly within the first few epochs.
If it is still below 0.1 after 10 epochs, stop the cell and tell Claude.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")
model.train(
    data=yaml_path,
    epochs=60,
    patience=15,
    imgsz=TILE,
    batch=16,
    device=0,
    flipud=0.5,
    project="/content/runs",
    name="pcb_tiles_v2",
    exist_ok=True,
)

save_dir = str(model.trainer.save_dir)
print("Training results saved in:", save_dir)

## Step 7: Measure how good the model is
We test on the tiles from the test photos, which the model has never seen (including all their empty tiles,
so false alarms on clean areas are counted).

- **Precision**: of the defects it reported, how many were real.
- **Recall**: of the real defects, how many it found.
- **mAP50**: one overall score between 0 and 1.

In [ ]:
best = YOLO(os.path.join(save_dir, "weights", "best.pt"))
metrics = best.val(data=yaml_path, split="test", imgsz=TILE)

summary = (
    f"Model: YOLOv8n fine-tuned on {TILE}x{TILE} tiles (stride {STRIDE}) of the PKU PCB dataset\n"
    f"Evaluated on {counts['test']['tiles']} tiles cut from {counts['test']['photos']} held-out test photos\n"
    f"Precision : {metrics.box.mp:.3f}\n"
    f"Recall    : {metrics.box.mr:.3f}\n"
    f"mAP50     : {metrics.box.map50:.3f}\n"
    f"mAP50-95  : {metrics.box.map:.3f}\n"
)
print(summary)

## Step 8: Convert the model for the Raspberry Pi
Converts `best.pt` (PyTorch format) to `best.onnx` (ONNX format), which is what the Pi runs.

In [ ]:
onnx_path = best.export(format="onnx", imgsz=TILE)
print("ONNX model:", onnx_path)

## Step 9: Pack everything into one zip and download it
The zip contains:
- `yolo_pcb.onnx`: the model for the Raspberry Pi
- `yolo_pcb.pt`: the same model in PyTorch format (keep it for later re-training)
- `metrics.txt`: the scores from Step 7
- the training charts for the report
- `sample_images/`: 12 full-size test photos (2 per defect class) to try in the dashboard

In [ ]:
from google.colab import files

out = "/content/pcb_model_v2"
shutil.rmtree(out, ignore_errors=True)
os.makedirs(os.path.join(out, "sample_images"))

shutil.copy(onnx_path, os.path.join(out, "yolo_pcb.onnx"))
shutil.copy(os.path.join(save_dir, "weights", "best.pt"), os.path.join(out, "yolo_pcb.pt"))

with open(os.path.join(out, "metrics.txt"), "w") as f:
    f.write(summary)

for pattern in ["*.png", "*.jpg", "results.csv"]:
    for p in glob.glob(os.path.join(save_dir, pattern)):
        shutil.copy(p, out)

picked = {c: 0 for c in range(len(CLASSES))}
for stem in sorted(s for s, split in split_of.items() if split == "test"):
    cls = labels[stem][0][0]
    if picked[cls] < 2:
        picked[cls] += 1
        shutil.copy(image_paths[stem], os.path.join(out, "sample_images"))

shutil.make_archive(out, "zip", out)
files.download(out + ".zip")